<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/04_train_pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device) # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)

Mounted at /content/drive
Thiết bị tính toán đang dùng: cpu


In [2]:
# ===== CELL 2: NẠP TOÀN BỘ DỮ LIỆU 8 ĐẶC TRƯNG TRỰC TIẾP LÊN VRAM GPU =====
import numpy as np  # Import NumPy để đọc file mảng .npy

# Nạp trực tiếp dữ liệu Train (8 inputs), Target (3 outputs) và cờ PhysMask lên GPU
X_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_train.npy')), dtype=torch.float32, device=device)
Y_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_train.npy')), dtype=torch.float32, device=device)
M_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'PhysMask_train.npy')), dtype=torch.float32, device=device)

# Nạp trực tiếp dữ liệu Validation lên GPU
X_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_validation.npy')), dtype=torch.float32, device=device)
Y_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_validation.npy')), dtype=torch.float32, device=device)

# Nạp trực tiếp dữ liệu Test lên GPU
X_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_test.npy')), dtype=torch.float32, device=device)
Y_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_test.npy')), dtype=torch.float32, device=device)

print(f'Đã nạp thẳng lên GPU ({device})! Input shape: {X_train_t.shape} | Mẫu hợp lệ vật lý: {int(M_train_t.sum())}')

Đã nạp thẳng lên GPU (cpu)! Input shape: torch.Size([89962, 8]) | Mẫu hợp lệ vật lý: 68607


In [3]:
# ===== CELL 3: KIẾN TRÚC PINN 8 ĐẦU VÀO & PHƯƠNG TRÌNH VI PHÂN ĐẦY ĐỦ =====
import torch.nn as nn  # Import module mạng Neural của PyTorch
import joblib          # Import joblib để đọc bộ chuẩn hóa Scaler

# Nạp lại X_scaler (8 chiều) và Y_scaler (3 chiều) để giải chuẩn hóa về đơn vị vật lý
X_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

# Đưa tham số chuẩn hóa lên GPU dưới dạng Tensor
x_mean = torch.tensor(X_scaler.mean_, dtype=torch.float32, device=device)
x_scale = torch.tensor(X_scaler.scale_, dtype=torch.float32, device=device)
y_mean = torch.tensor(Y_scaler.mean_, dtype=torch.float32, device=device)
y_scale = torch.tensor(Y_scaler.scale_, dtype=torch.float32, device=device)

class PMSM_PINN(nn.Module):
    """
    Kiến trúc mạng PINN nhận 8 đặc trưng đầu vào [ud, uq, Id, Iq, nm, dId_dt, dIq_dt, dnm_dt]
    và dự đoán 3 tham số động cơ [Rs, Ld, Lq].
    """
    def __init__(self, input_dim=8, output_dim=3, hidden_dim=128):
        super(PMSM_PINN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),  # Hàm kích hoạt SiLU (Swish) đạo hàm trơn hơn Tanh, giúp hội tụ tốt với bài toán vi phân
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

def compute_physics_loss(inputs_scaled, outputs_scaled, phys_mask, pole_pairs=4, psi_m=0.527139):
    """
    Hàm tính sai số vật lý dựa trên HỆ PHƯƠNG TRÌNH VI PHÂN ĐẦY ĐỦ của động cơ PMSM (bao gồm cả thành phần quá độ L*di/dt):
      ud = Rs * Id + Ld * (dId/dt) - omega_e * Lq * Iq
      uq = Rs * Iq + Lq * (dIq/dt) + omega_e * (Ld * Id + psi_m)
    """
    # Lọc lấy các mẫu vận hành chuẩn (phys_mask == 1.0)
    mask = phys_mask > 0.5
    if mask.sum() == 0:
        return torch.tensor(0.0, device=inputs_scaled.device)

    in_s = inputs_scaled[mask]
    out_s = outputs_scaled[mask]

    # Giải chuẩn hóa về đơn vị vật lý thực tế
    inputs_real = in_s * x_scale + x_mean
    outputs_real = out_s * y_scale + y_mean

    # Tách 8 biến đầu vào thực tế
    ud = inputs_real[:, 0]      # Điện áp trục d (V)
    uq = inputs_real[:, 1]      # Điện áp trục q (V)
    Id = inputs_real[:, 2]      # Dòng điện trục d (A)
    Iq = inputs_real[:, 3]      # Dòng điện trục q (A)
    nm = inputs_real[:, 4]      # Tốc độ cơ học (rpm)
    dId_dt = inputs_real[:, 5]  # Đạo hàm dòng trục d theo thời gian (A/s)
    dIq_dt = inputs_real[:, 6]  # Đạo hàm dòng trục q theo thời gian (A/s)

    # Tách 3 tham số do mạng PINN ước lượng
    Rs_est = outputs_real[:, 0]  # Điện trở stator ước lượng (Ohm)
    Ld_est = outputs_real[:, 1]  # Điện cảm trục d ước lượng (H)
    Lq_est = outputs_real[:, 2]  # Điện cảm trục q ước lượng (H)

    # Quy đổi tốc độ cơ học nm (rpm) sang tốc độ góc điện omega_e (rad/s)
    omega_m = nm * (2.0 * np.pi / 60.0)
    omega_e = pole_pairs * omega_m

    # Tính điện áp ud, uq lý thuyết theo phương trình vi phân đầy đủ (cả quá độ lẫn xác lập)
    ud_calc = Rs_est * Id + Ld_est * dId_dt - omega_e * Lq_est * Iq
    uq_calc = Rs_est * Iq + Lq_est * dIq_dt + omega_e * (Ld_est * Id + psi_m)

    # Dùng SmoothL1Loss (Huber Loss) để chống nhiễu xung PWM cục bộ tốt hơn MSE thuần túy
    loss_ud = nn.functional.smooth_l1_loss(ud_calc, ud, beta=2.0)
    loss_uq = nn.functional.smooth_l1_loss(uq_calc, uq, beta=2.0)

    return loss_ud + loss_uq

# Khởi tạo mô hình PINN 8 đầu vào trên GPU
model_pinn = PMSM_PINN(input_dim=8, output_dim=3, hidden_dim=128).to(device)
print("Đã khởi tạo mô hình PINN phương trình vi phân đầy đủ trên thiết bị:", device)

Đã khởi tạo mô hình PINN phương trình vi phân đầy đủ trên thiết bị: cpu


In [4]:
# ===== CELL 4: HUẤN LUYỆN PINN (CHỐNG OVERFITTING & CÂN BẰNG TRỌNG SỐ) =====
import torch.optim as optim  # Import bộ tối ưu hóa trọng số
import shutil                # Import thư viện copy file
import time                  # Import thư viện đo thời gian

# Dùng SmoothL1Loss cho dữ liệu để tránh bị kéo lệch bởi các điểm nhảy bậc Rs đột ngột
criterion_data = nn.SmoothL1Loss(beta=0.5)
# Dùng AdamW với weight_decay=1e-4 để chống học vẹt tập Train, giúp giảm Val Loss và Test Error
optimizer = optim.AdamW(model_pinn.parameters(), lr=0.0015, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

lambda_p = 0.05              # Tăng trọng số Physics Loss lên 0.05 vì sai số vật lý giờ đã nhỏ và chuẩn xác
epochs = 150                 # Tổng số vòng lặp huấn luyện
batch_size = 512             # Kích thước mỗi lô dữ liệu xử lý trên GPU
n_train = len(X_train_t)     # Tổng số mẫu Train
best_val_loss = float('inf')

LOCAL_CKPT = '/content/best_pinn_model.pth'
DRIVE_CKPT = os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth')

print("=== BẮT ĐẦU HUẤN LUYỆN PINN (FULL ODE PHYSICS) ===")
start_time = time.time()

for epoch in range(1, epochs + 1):
    # --- GIAI ĐOẠN 1: HUẤN LUYỆN (TRAIN) ---
    model_pinn.train()
    perm = torch.randperm(n_train, device=device)  # Xáo trộn dữ liệu ngẫu nhiên trên GPU

    train_data_loss = 0.0
    train_phys_loss = 0.0

    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        inputs = X_train_t[idx]
        targets = Y_train_t[idx]
        pmask = M_train_t[idx]

        optimizer.zero_grad()
        outputs = model_pinn(inputs)

        loss_d = criterion_data(outputs, targets)               # Sai số dữ liệu
        loss_p = compute_physics_loss(inputs, outputs, pmask)   # Sai số phương trình vi phân vật lý
        loss_total = loss_d + lambda_p * loss_p                 # Tổng hợp Loss

        loss_total.backward()
        optimizer.step()

        train_data_loss += loss_d.item() * len(idx)
        train_phys_loss += loss_p.item() * len(idx)

    train_data_loss /= n_train
    train_phys_loss /= n_train

    # --- GIAI ĐOẠN 2: ĐÁNH GIÁ TRÊN VALIDATION ---
    model_pinn.eval()
    with torch.no_grad():
        val_outputs = model_pinn(X_val_t)
        val_loss = criterion_data(val_outputs, Y_val_t).item()

    scheduler.step(val_loss)

    # Lưu mô hình có Val Loss thấp nhất
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_pinn.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        }, LOCAL_CKPT)

    if epoch % 15 == 0 or epoch == 1:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch:3d}/{epochs}] | Data Loss: {train_data_loss:.6f} | Phys Loss: {train_phys_loss:.4f} | Val Loss: {val_loss:.6f} | Thời gian: {elapsed:.1f} giây")

# Copy trọng số tốt nhất sang Google Drive
shutil.copy(LOCAL_CKPT, DRIVE_CKPT)
print(f"\n--> Hoàn thành 150 epochs trong {time.time() - start_time:.1f} giây! Đã lưu vào: {DRIVE_CKPT}")

=== BẮT ĐẦU HUẤN LUYỆN PINN (FULL ODE PHYSICS) ===
Epoch [  1/150] | Data Loss: 0.213149 | Phys Loss: 4.6480 | Val Loss: 0.352185 | Thời gian: 1.0 giây
Epoch [ 15/150] | Data Loss: 0.127884 | Phys Loss: 4.6506 | Val Loss: 0.369141 | Thời gian: 13.3 giây
Epoch [ 30/150] | Data Loss: 0.115367 | Phys Loss: 4.6522 | Val Loss: 0.391313 | Thời gian: 26.8 giây
Epoch [ 45/150] | Data Loss: 0.112051 | Phys Loss: 4.6494 | Val Loss: 0.394218 | Thời gian: 40.2 giây
Epoch [ 60/150] | Data Loss: 0.110472 | Phys Loss: 4.6493 | Val Loss: 0.379197 | Thời gian: 53.4 giây
Epoch [ 75/150] | Data Loss: 0.109384 | Phys Loss: 4.6491 | Val Loss: 0.380396 | Thời gian: 66.8 giây
Epoch [ 90/150] | Data Loss: 0.108795 | Phys Loss: 4.6498 | Val Loss: 0.383651 | Thời gian: 79.5 giây
Epoch [105/150] | Data Loss: 0.108444 | Phys Loss: 4.6493 | Val Loss: 0.382083 | Thời gian: 93.2 giây
Epoch [120/150] | Data Loss: 0.108284 | Phys Loss: 4.6492 | Val Loss: 0.383990 | Thời gian: 106.9 giây
Epoch [135/150] | Data Loss: 0.

In [5]:
# ===== CELL 5: ĐÁNH GIÁ PINN TRÊN TẬP TEST =====

# Nạp bản trọng số tốt nhất đã lưu trên Google Drive ra để đánh giá
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_pinn_model.pth'))
model_pinn.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trạng thái trọng số tối ưu
model_pinn.eval()                                          # Bật chế độ dự đoán

with torch.no_grad():  # Tắt tính toán gradient khi kiểm tra tập Test
    # Dự đoán toàn bộ tập Test trực tiếp trên GPU và chuyển kết quả về CPU dạng mảng NumPy
    all_preds = model_pinn(X_test_t).cpu().numpy()
    all_targets = Y_test_t.cpu().numpy()

# Giải chuẩn hóa (Inverse Transform) đưa kết quả về đơn vị vật lý thực (Ohm cho Rs, Henry cho Ld và Lq)
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số Rs, Ld, Lq
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình (PINN): {mape[0]:.2f}%") # In % sai số của điện trở stator Rs
print(f"Sai số Ld trung bình (PINN): {mape[1]:.2f}%") # In % sai số của độ tự cảm trục d Ld
print(f"Sai số Lq trung bình (PINN): {mape[2]:.2f}%") # In % sai số của độ tự cảm trục q Lq

=== KẾT QUẢ ĐÁNH GIÁ PINN TRÊN TẬP TEST ===
Sai số Rs trung bình (PINN): 8.22%
Sai số Ld trung bình (PINN): 0.15%
Sai số Lq trung bình (PINN): 0.15%
